# Shopee VN vs ID Research Notebook

Notebook này vẽ biểu đồ để kiểm tra các giả thuyết trong `research.md`.

Input chính: `Dataset/DataProcessed/product_dataset_ready.csv`


In [ ]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 120)

DATA_PATH = Path('Dataset/DataProcessed/product_dataset_ready.csv')
df = pd.read_csv(DATA_PATH)
df.shape


In [ ]:
numeric_cols = [
    'price_num', 'price_original_num', 'price_before_promo_num', 'discount_percent_num',
    'voucher_discount_num', 'voucher_min_spend_num', 'monthly_sold_value_num',
    'history_sold_value_num', 'rating_num', 'rating_count_num', 'liked_count_num',
    'images_count', 'vouchers_count', 'tier_variation_options_count', 'shop_category_count',
    'shop_follower_count', 'shop_item_count', 'shop_rating_star', 'shop_response_rate',
    'shop_response_time', 'shop_rating_good', 'shop_rating_normal', 'shop_rating_bad',
    'shop_cancellation_rate'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['estimated_recent_revenue'] = df['price_num'] * df['monthly_sold_value_num']
df['has_voucher'] = df['voucher_discount_num'].fillna(0) > 0
df['has_promo'] = df['discount_percent_num'].fillna(0) > 0

def promo_group(row):
    if row['has_voucher'] and row['has_promo']:
        return 'voucher + promo'
    if row['has_voucher']:
        return 'voucher only'
    if row['has_promo']:
        return 'promo only'
    return 'no voucher/promo'

df['promo_group'] = df.apply(promo_group, axis=1)
df['product_name_len'] = df['product_name'].fillna('').str.len()
df['has_brand'] = df['brand'].fillna('').str.strip().ne('')
df['is_official_shop'] = df['shop_is_official_shop'].astype(str).str.lower().eq('true')

df['discount_bucket'] = pd.cut(
    df['discount_percent_num'].fillna(0),
    bins=[-0.1, 0, 10, 20, 40, 60, 100],
    labels=['0%', '1-10%', '11-20%', '21-40%', '41-60%', '61-100%']
)

df[['country_code', 'price_num', 'monthly_sold_value_num', 'estimated_recent_revenue', 'promo_group']].head()


## 1. Market Comparison: VN vs ID


In [ ]:
country_summary = df.groupby('country_code').agg(
    products=('item_id', 'count'),
    shops=('shop_id', 'nunique'),
    median_price=('price_num', 'median'),
    median_monthly_sold=('monthly_sold_value_num', 'median'),
    total_estimated_revenue=('estimated_recent_revenue', 'sum'),
    avg_rating=('rating_num', 'mean'),
    voucher_rate=('has_voucher', 'mean'),
    promo_rate=('has_promo', 'mean')
).reset_index()
country_summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=country_summary, x='country_code', y='total_estimated_revenue', ax=axes[0])
axes[0].set_title('Total Estimated Recent Revenue by Country')
axes[0].set_ylabel('price * monthly_sold_value')

sns.boxplot(data=df, x='country_code', y='price_num', ax=axes[1], showfliers=False)
axes[1].set_title('Price Distribution by Country')

sns.boxplot(data=df, x='country_code', y='monthly_sold_value_num', ax=axes[2], showfliers=False)
axes[2].set_title('Monthly Sold Distribution by Country')

plt.tight_layout()


## 2. Promotion/Voucher Effectiveness


In [ ]:
promo_summary = df.groupby(['country_code', 'promo_group']).agg(
    products=('item_id', 'count'),
    median_monthly_sold=('monthly_sold_value_num', 'median'),
    median_revenue=('estimated_recent_revenue', 'median'),
    total_revenue=('estimated_recent_revenue', 'sum'),
    median_discount=('discount_percent_num', 'median'),
    median_voucher=('voucher_discount_num', 'median')
).reset_index()
promo_summary.sort_values(['country_code', 'total_revenue'], ascending=[True, False])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
order = ['no voucher/promo', 'promo only', 'voucher only', 'voucher + promo']
sns.boxplot(data=df, x='promo_group', y='monthly_sold_value_num', hue='country_code', order=order, ax=axes[0], showfliers=False)
axes[0].set_title('Monthly Sold by Promotion Group')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=promo_summary, x='promo_group', y='median_revenue', hue='country_code', order=order, ax=axes[1])
axes[1].set_title('Median Estimated Revenue by Promotion Group')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()


In [ ]:
discount_summary = df.groupby(['country_code', 'discount_bucket'], observed=True).agg(
    products=('item_id', 'count'),
    median_monthly_sold=('monthly_sold_value_num', 'median'),
    median_revenue=('estimated_recent_revenue', 'median')
).reset_index()

plt.figure(figsize=(12, 5))
sns.barplot(data=discount_summary, x='discount_bucket', y='median_revenue', hue='country_code')
plt.title('Median Estimated Revenue by Discount Bucket')
plt.xlabel('Discount Bucket')
plt.ylabel('Median Estimated Revenue')
plt.tight_layout()


## 3. Revenue Drivers


In [ ]:
feature_cols = [
    'price_num', 'discount_percent_num', 'voucher_discount_num', 'monthly_sold_value_num',
    'rating_num', 'rating_count_num', 'liked_count_num', 'images_count',
    'tier_variation_options_count', 'shop_category_count', 'shop_follower_count',
    'shop_rating_star', 'shop_response_rate', 'shop_response_time', 'product_name_len',
    'estimated_recent_revenue'
]
corr = df[feature_cols].corr(numeric_only=True)

plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=False)
plt.title('Correlation Heatmap')
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=df, x='price_num', y='monthly_sold_value_num', hue='country_code', alpha=0.5, ax=axes[0])
axes[0].set_title('Price vs Monthly Sold')

sns.scatterplot(data=df, x='liked_count_num', y='monthly_sold_value_num', hue='country_code', alpha=0.5, ax=axes[1])
axes[1].set_title('Liked Count vs Monthly Sold')

sns.scatterplot(data=df, x='rating_count_num', y='estimated_recent_revenue', hue='country_code', alpha=0.5, ax=axes[2])
axes[2].set_title('Rating Count vs Estimated Revenue')

for ax in axes:
    ax.set_xscale('symlog')
    ax.set_yscale('symlog')

plt.tight_layout()


## 4. Shop Trust and Shop Performance


In [ ]:
shop_summary = df.groupby(['country_code', 'shop_id', 'shop_name', 'is_official_shop']).agg(
    products=('item_id', 'count'),
    total_revenue=('estimated_recent_revenue', 'sum'),
    median_monthly_sold=('monthly_sold_value_num', 'median'),
    shop_follower_count=('shop_follower_count', 'max'),
    shop_rating_star=('shop_rating_star', 'max')
).reset_index().sort_values('total_revenue', ascending=False)
shop_summary.head(15)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=df, x='is_official_shop', y='estimated_recent_revenue', hue='country_code', estimator=np.median, ax=axes[0])
axes[0].set_title('Median Estimated Revenue: Official vs Non-official')

sns.scatterplot(data=shop_summary, x='shop_follower_count', y='total_revenue', hue='country_code', style='is_official_shop', s=90, ax=axes[1])
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title('Shop Followers vs Total Estimated Revenue')

plt.tight_layout()


## 5. Category and Merchandising


In [ ]:
category_rows = df.assign(shop_category_names=df['shop_category_names'].fillna(''))
category_rows = category_rows.assign(shop_category_name=category_rows['shop_category_names'].str.split('|'))
category_rows = category_rows.explode('shop_category_name')
category_rows['shop_category_name'] = category_rows['shop_category_name'].fillna('').str.strip()
category_rows = category_rows[category_rows['shop_category_name'].ne('')]

cat_summary = category_rows.groupby(['country_code', 'shop_category_name']).agg(
    products=('item_id', 'nunique'),
    total_revenue=('estimated_recent_revenue', 'sum'),
    median_monthly_sold=('monthly_sold_value_num', 'median')
).reset_index().sort_values('total_revenue', ascending=False)
cat_summary.head(20)


In [ ]:
top_cat = cat_summary.head(15).copy()
plt.figure(figsize=(12, 7))
sns.barplot(data=top_cat, y='shop_category_name', x='total_revenue', hue='country_code', dodge=False)
plt.title('Top Internal Shop Categories by Estimated Revenue')
plt.xlabel('Estimated Revenue')
plt.ylabel('Shop Category')
plt.tight_layout()


In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='shop_category_count', y='monthly_sold_value_num', hue='country_code', showfliers=False)
plt.title('Monthly Sold by Number of Internal Shop Categories')
plt.tight_layout()


## 6. Product Presentation: Images, Brand, Keywords


In [ ]:
keywords = ['combo', 'official', 'sale', 'new', 'best seller', 'khuyến mãi', 'flash', 'gift']
for kw in keywords:
    df[f'kw_{kw}'] = df['product_name'].fillna('').str.lower().str.contains(kw, regex=False)

keyword_summary = []
for kw in keywords:
    col = f'kw_{kw}'
    tmp = df.groupby(['country_code', col]).agg(
        products=('item_id', 'count'),
        median_monthly_sold=('monthly_sold_value_num', 'median'),
        median_revenue=('estimated_recent_revenue', 'median')
    ).reset_index()
    tmp['keyword'] = kw
    tmp = tmp[tmp[col] == True].drop(columns=[col])
    keyword_summary.append(tmp)

keyword_summary = pd.concat(keyword_summary, ignore_index=True)
keyword_summary.sort_values('median_revenue', ascending=False)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=df, x='images_count', y='monthly_sold_value_num', hue='country_code', showfliers=False, ax=axes[0])
axes[0].set_title('Monthly Sold by Image Count')

sns.barplot(data=df, x='has_brand', y='estimated_recent_revenue', hue='country_code', estimator=np.median, ax=axes[1])
axes[1].set_title('Median Estimated Revenue: Has Brand vs No Brand')

sns.barplot(data=keyword_summary, x='keyword', y='median_revenue', hue='country_code', ax=axes[2])
axes[2].set_title('Median Revenue by Product-name Keyword')
axes[2].tick_params(axis='x', rotation=35)

plt.tight_layout()


## 7. Suggested Next Steps

- Loại hoặc winsorize outlier như `price = 999999999` trước khi modeling.
- Dùng log transform cho `price`, `monthly_sold_value`, `estimated_recent_revenue` vì phân phối lệch mạnh.
- Tách phân tích theo từng nước trước, sau đó mới so sánh cross-country.
- Khi phân tích category nội bộ, chú ý double count vì một sản phẩm có thể nằm trong nhiều category.
- Nếu cần model hóa, bắt đầu bằng regression/tree model với target `log1p(estimated_recent_revenue)`.
